# Session 2.5 — Lab: scaling laws and a first look inside a model

**African Technical AI Safety** · Week 1, Session 2.5

This notebook works through the three parts of the lab:

1. **Fit a scaling law** — recover a power-law exponent from (compute, loss) points and compare it with Kaplan et al.
2. **Explore the compute trends** — estimate a doubling time from Epoch AI's notable-models data.
3. **Open up a transformer** — load GPT-2 in TransformerLens, read the residual stream, and measure the tokenisation gap between English and isiZulu.

Everything runs on free Colab; no GPU is needed, though a free GPU runtime makes part 3 quicker.

**What you submit:** this notebook, run end to end, with your own answers written into the four *Your turn* cells.

> Cells marked **Your turn** are where you write. The code cells are meant to run as they are; break them deliberately if it helps you understand them, but keep a working copy of your answers.

In [ ]:
# Run once per Colab session (~1 min). Skip the install if running locally with the packages already present.
import importlib.util, sys

IN_COLAB = 'google.colab' in sys.modules
if importlib.util.find_spec('transformer_lens') is None:
    %pip install -q transformer_lens

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Colab:', IN_COLAB)

---
## ① Fit a scaling law

Kaplan et al. (2020) report that test loss falls as a power law in compute:

$$L(C_{\min}) = \left(\frac{C_c^{\min}}{C_{\min}}\right)^{\alpha_C^{\min}}, \qquad \alpha_C^{\min} \approx 0.050, \quad C_c^{\min} \approx 3.1 \times 10^8 \ \text{PF-days}$$

Taking logs turns that into a straight line, which is why the fit below is ordinary least squares:

$$\log L = a - \alpha \log C$$

Your job is to recover $\alpha$ from data and see how small it is.

In [ ]:
# The class CSV, if your instructor distributed one: two columns, compute and loss.
# Set CSV_PATH to its filename after uploading it to the Colab session.
CSV_PATH = None   # e.g. 'scaling_runs.csv'

ALPHA_KAPLAN, CC_KAPLAN = 0.050, 3.1e8   # Kaplan et al. (2020), eq. 1.3; C in PF-days

if CSV_PATH:
    df = pd.read_csv(CSV_PATH)
    compute, loss = df.iloc[:, 0].to_numpy(), df.iloc[:, 1].to_numpy()
    source = f'class CSV ({CSV_PATH})'
else:
    # No CSV: generate teaching data FROM the published law, with multiplicative
    # noise standing in for run-to-run variation. These are not measured runs, so
    # recovering alpha here only checks your fitting code, not the law itself.
    rng = np.random.default_rng(0)
    compute = np.logspace(-8, 2, 18)                       # PF-days
    loss = (CC_KAPLAN / compute) ** ALPHA_KAPLAN * np.exp(rng.normal(0, 0.015, compute.size))
    source = 'generated from Kaplan et al. eq. 1.3 (synthetic, not measured)'

print(f'{len(compute)} points from {source}')
print(f'compute spans {compute.min():.1e} to {compute.max():.1e} PF-days')

In [ ]:
# Plot on log-log axes: a power law is a straight line here, and nowhere else.
fig, ax = plt.subplots()
ax.loglog(compute, loss, 'o', color='#2a5298')
ax.set_xlabel('Training compute $C$ (PF-days)')
ax.set_ylabel('Test loss $L$')
ax.set_title('Loss against compute, log–log')
plt.show()

In [ ]:
# Least-squares fit of log10(L) = a - alpha * log10(C).
slope, intercept = np.polyfit(np.log10(compute), np.log10(loss), 1)
alpha = -slope

print(f'fitted exponent alpha = {alpha:.4f}')
print(f'Kaplan et al.         = {ALPHA_KAPLAN:.4f}   (their alpha_C^min)')
print(f'for reference, their other two exponents: alpha_N ~ 0.076, alpha_D ~ 0.095')
print()
print(f'A 10x increase in compute multiplies the loss by 10^-alpha = {10 ** -alpha:.3f},')
print(f'i.e. it buys a {100 * (1 - 10 ** -alpha):.1f}% reduction. The exponent is small: this is')
print('why the curve is so punishing, and why frontier runs cost what they do.')

fig, ax = plt.subplots()
ax.loglog(compute, loss, 'o', color='#2a5298', label='data')
grid = np.logspace(np.log10(compute.min()), np.log10(compute.max()), 100)
ax.loglog(grid, 10 ** (intercept + slope * np.log10(grid)), '-', color='#003A70',
          label=f'fit: slope $-{alpha:.3f}$')
ax.set_xlabel('Training compute $C$ (PF-days)'); ax.set_ylabel('Test loss $L$')
ax.legend(); ax.set_title('Fitted power law')
plt.show()

In [ ]:
# Extrapolate one order of magnitude past the largest run in the data.
C_far = compute.max() * 10
L_far = 10 ** (intercept + slope * np.log10(C_far))

print(f'largest run in the data: C = {compute.max():.2e} PF-days, L = {loss[-1]:.3f}')
print(f'extrapolated one OOM out: C = {C_far:.2e} PF-days, L = {L_far:.3f}')
print(f'predicted improvement: {100 * (1 - L_far / loss[-1]):.1f}%')

### Your turn (①)

Write **one sentence** on why you should distrust your own extrapolation. Session 2.4 is the
place to look: what does a smooth loss curve fail to tell you about the capabilities that appear
along it, and what happened to Kaplan's own compute-optimal advice when Chinchilla re-ran the
experiment more carefully?

*Your answer:*


---
## ② Explore the compute trends

Epoch AI maintain a public dataset of notable AI models with, among much else, a publication date
and an estimate of training compute. We plot compute against date on a log axis and read off a
doubling time.

Note what this measurement is and is not. Training compute is an *input*, not a capability, and the
estimates are reconstructed from papers and reports of varying candour.

In [ ]:
EPOCH_CSV = 'https://epoch.ai/data/notable_ai_models.csv'

try:
    models = pd.read_csv(EPOCH_CSV, low_memory=False)
    print(f'loaded {len(models)} rows from epoch.ai')
except Exception as e:
    print('Download failed:', e)
    print('Fetch the CSV by hand from https://epoch.ai/data/notable-ai-models')
    print('("Download the data in CSV"), upload it to this session, and read it here.')
    models = None

if models is not None:
    print('columns we need:', [c for c in models.columns if c in
                               ('Model', 'Publication date', 'Training compute (FLOP)')])

In [ ]:
# Keep the rows that have both a date and a compute estimate.
d = models[['Model', 'Publication date', 'Training compute (FLOP)']].dropna().copy()
d['Publication date'] = pd.to_datetime(d['Publication date'], errors='coerce')
d = d.dropna().sort_values('Publication date')

START = '2010-01-01'   # the deep-learning era; try 2018 or 2020 and watch the answer move
era = d[d['Publication date'] >= START]
print(f'{len(era)} models from {START} onwards, out of {len(d)} with usable data')
print(f'earliest: {era.iloc[0]["Model"]} ({era.iloc[0]["Publication date"].date()})')
print(f'latest:   {era.iloc[-1]["Model"]} ({era.iloc[-1]["Publication date"].date()})')

In [ ]:
# Fit log10(compute) against time in years, then convert the slope to a doubling time.
years = (era['Publication date'] - pd.Timestamp(START)).dt.days / 365.25
logC = np.log10(era['Training compute (FLOP)'])
slope_yr, icept_yr = np.polyfit(years, logC, 1)
doubling_months = np.log10(2) / slope_yr * 12

print(f'slope: {slope_yr:.3f} orders of magnitude per year')
print(f'doubling time: {doubling_months:.1f} months')
print(f'that is {10 ** slope_yr:.1f}x per year')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(era['Publication date'], era['Training compute (FLOP)'], s=14, alpha=0.55,
           color='#2a5298', label='notable models')
line_x = pd.to_datetime([era['Publication date'].min(), era['Publication date'].max()])
line_years = (line_x - pd.Timestamp(START)).days / 365.25
ax.plot(line_x, 10 ** (icept_yr + slope_yr * line_years), color='#003A70',
        label=f'fit: doubling every {doubling_months:.1f} months')
ax.set_yscale('log'); ax.set_xlabel('Publication date'); ax.set_ylabel('Training compute (FLOP)')
ax.set_title(f'Training compute of notable AI models, {START[:4]} onwards')
ax.legend(); plt.show()

In [ ]:
# The fit is not the whole story: label the models at the top of the range and see
# how much of the trend rests on a handful of frontier runs.
top = era.nlargest(8, 'Training compute (FLOP)')[['Model', 'Publication date', 'Training compute (FLOP)']]
top['Publication date'] = top['Publication date'].dt.date
print(top.to_string(index=False))

### Your turn (②)

Two sentences, one each:

1. **Which quantity did you actually measure?** Be precise: not "AI progress", but the thing on the y-axis and how it was estimated.
2. **Over what window, and what would change your answer?** Re-run with `START = '2018-01-01'` and `'2020-01-01'` before you write this.

This is the Session 1.4 reader's checklist applied to a plot you made yourself, which is the harder case.

*Your answers:*


---
## ③ Open up a transformer

First contact with the residual-stream picture from 2.2, on a real model. GPT-2 small: 12 layers,
12 heads, $d_{\text{model}} = 768$.

The first cell downloads the weights (about 500 MB) and takes a minute or two. CPU is fine.

In [ ]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained('gpt2')
print(f'layers: {model.cfg.n_layers} | heads/layer: {model.cfg.n_heads} | '
      f'd_model: {model.cfg.d_model} | vocab: {model.cfg.d_vocab}')

In [ ]:
# Feed it some prompts and read off the top predicted next tokens.
prompts = [
    'The Eiffel Tower is in the city of',
    'The capital city of South Africa is',
    'The largest city in Africa is',
    'The answer to 17 plus 25 is',
]

for p in prompts:
    logits = model(p)                       # [batch, position, vocab]
    probs = logits[0, -1].softmax(dim=-1)   # distribution over the NEXT token only
    top_p, top_i = probs.topk(5)
    print(f'\n{p!r}')
    for prob, idx in zip(top_p, top_i):
        print(f'   {model.to_string(idx.item())!r:>16}  {prob.item():.3f}')
    print(f'   entropy: {-(probs * probs.log()).sum().item():.2f} nats')

Where is it confident and where is it not? Entropy is the number to watch: a peaked distribution
(low entropy) means the model has effectively decided. Note that confidence and correctness are
different things, which is the whole reason Session 1.4 has a checklist.

In [ ]:
# run_with_cache captures every intermediate activation.
text = 'The Eiffel Tower is in the city of'
logits, cache = model.run_with_cache(text)

resid = cache['resid_post', 0]     # residual stream after block 0
print(f'residual stream shape: {tuple(resid.shape)}   # [batch, position, d_model]')
print(f'tokens: {model.to_str_tokens(text)}')
print()
print('This is the T x d matrix from 2.2: one row per token position, one column per')
print('residual-stream dimension. Every block reads from it and writes back into it.')
print()
print('a few other things the cache holds:')
for key in ['blocks.0.attn.hook_pattern', 'blocks.0.hook_mlp_out', 'ln_final.hook_normalized']:
    if key in cache:
        print(f'   {key:<34} {tuple(cache[key].shape)}')
    else:
        print(f'   {key:<34} (not in this build; try list(cache.keys())[:20])')
print()
print('attention pattern is [batch, head, query_pos, key_pos]: for each head, how much')
print('each token attends to each earlier token.')

In [ ]:
# How the residual stream grows as it passes through the blocks.
norms = [cache['resid_post', l][0].norm(dim=-1).mean().item() for l in range(model.cfg.n_layers)]
fig, ax = plt.subplots()
ax.plot(range(model.cfg.n_layers), norms, 'o-', color='#2a5298')
ax.set_xlabel('block'); ax.set_ylabel('mean residual-stream norm')
ax.set_title('The residual stream accumulates as it goes')
plt.show()

### The tokenisation check (the African-safety thread)

GPT-2's tokenizer is a byte-pair-encoding vocabulary of 50,257 tokens fitted mostly to English text.
The same sentence in a low-resource language is therefore chopped into more pieces.

To measure this honestly we need *parallel* text: the same meaning in both languages. We use
[MAFAND-MT](https://github.com/masakhane-io/lafand-mt), the Masakhane news translation dataset
(Adelani et al., 2022; dataset CC BY-NC 4.0), downloaded from the source rather than reproduced here.

In [ ]:
MAFAND = 'https://raw.githubusercontent.com/masakhane-io/lafand-mt/main/data/tsv_files/en-{lang}/dev.tsv'

try:
    pairs = pd.read_csv(MAFAND.format(lang='zul'), sep='\t').dropna()
    print(f'{len(pairs)} parallel English–isiZulu sentence pairs')
    example = pairs.iloc[1]
except Exception as e:
    print('Download failed:', e)
    print('Substitute your own sentence pair below: any language you know, same meaning in both.')
    pairs, example = None, {'en': 'Type an English sentence here.',
                            'zul': 'Type the same sentence in your language here.'}

for lang, sentence in [('English', example['en']), ('isiZulu', example['zul'])]:
    toks = model.to_str_tokens(sentence)
    print(f'\n{lang}: {sentence}')
    print(f'   {len(toks)} tokens, {len(sentence)} characters')
    print(f'   {toks}')

In [ ]:
# Across many sentence pairs, not one: the ratio is the number that matters.
if pairs is not None:
    sample = pairs.head(200)
    en_tok = sample['en'].apply(lambda s: len(model.to_str_tokens(s)))
    zu_tok = sample['zul'].apply(lambda s: len(model.to_str_tokens(s)))
    en_ch, zu_ch = sample['en'].str.len(), sample['zul'].str.len()

    print(f'over {len(sample)} parallel sentences:')
    print(f'   mean tokens, English: {en_tok.mean():.1f}')
    print(f'   mean tokens, isiZulu: {zu_tok.mean():.1f}')
    print(f'   token ratio (isiZulu / English): {zu_tok.mean() / en_tok.mean():.2f}x')
    print()
    print('Characters, to check the ratio is not just longer words:')
    print(f'   tokens per 100 chars, English: {100 * en_tok.sum() / en_ch.sum():.1f}')
    print(f'   tokens per 100 chars, isiZulu: {100 * zu_tok.sum() / zu_ch.sum():.1f}')

    fig, ax = plt.subplots()
    ax.hist(zu_tok / en_tok, bins=30, color='#2a5298', alpha=0.85)
    ax.axvline(1.0, color='#555', ls='--', label='parity')
    ax.set_xlabel('tokens in isiZulu ÷ tokens in English, per sentence pair')
    ax.set_ylabel('sentence pairs'); ax.legend()
    ax.set_title('The same meaning costs more tokens in isiZulu')
    plt.show()

In [ ]:
# Optional: the same measurement across several African languages.
# MAFAND covers amh, hau, ibo, kin, lug, luo, nya, pcm, sna, swa, tsn, twi, xho, yor, zul.
LANGS = ['zul', 'xho', 'yor', 'swa', 'hau']

rows = []
for lang in LANGS:
    try:
        p = pd.read_csv(MAFAND.format(lang=lang), sep='\t').dropna().head(150)
        col = [c for c in p.columns if c != 'en'][0]
        e = p['en'].apply(lambda s: len(model.to_str_tokens(s))).mean()
        o = p[col].apply(lambda s: len(model.to_str_tokens(s))).mean()
        rows.append({'language': lang, 'en tokens': round(e, 1),
                     'lang tokens': round(o, 1), 'ratio': round(o / e, 2)})
    except Exception as err:
        print(f'{lang}: skipped ({err})')

if rows:
    print(pd.DataFrame(rows).sort_values('ratio', ascending=False).to_string(index=False))

### Your turn (③)

Write a short paragraph on what the ratio you measured implies. Cover three things:

1. **Cost**: APIs bill per token and context windows are counted in tokens. What does a ratio of, say, 2x mean for a user in Durban against a user in London?
2. **Quality**: more tokens per word means the model sees the language in smaller, less meaningful fragments. Why would that hurt performance even before any safety question arises?
3. **Safety**: connect this to the Session 1.4 anchor. If a language is under-represented enough to tokenise badly, what does that predict about the amount of safety training and red-teaming it received? (Sessions 9 and 18 return to this.)

*Your answer:*


---
## Submission checklist

- [ ] ① fitted exponent, and your one-sentence extrapolation caveat
- [ ] ② compute-trend plot, and your two sentences on what you measured and over what window
- [ ] ③ GPT-2 top-token outputs, the residual-stream shape, the tokenisation comparison, and your paragraph

Save the notebook with its outputs intact (Colab: *File → Download → .ipynb*) and submit it.
Graded on completion and correctness; resubmission is allowed, because the point is mastery
rather than one-shot performance.

**Sources used here.** Kaplan et al. (2020), [arXiv:2001.08361](https://arxiv.org/abs/2001.08361), for the
power law and its exponents. [Epoch AI](https://epoch.ai/data/notable-ai-models) for the compute trend data.
[TransformerLens](https://github.com/TransformerLensOrg/TransformerLens) (MIT) for model access.
[MAFAND-MT](https://github.com/masakhane-io/lafand-mt) (Adelani et al., 2022; CC BY-NC 4.0) for the
parallel sentences.